# Benchmark: local models on a fixed article set
Same 16 frozen articles, same prompts; only the model changes.
**Runtime → Change runtime type → T4 GPU.**

Two configurations:
- **A. fixed verifier** (Qwen2.5-7B fact-checks for every model): isolates *generation* quality. ~35 min.
- **B. self-verification** (each model checks its own facts): what the pipeline does by default. ~35 min.

Run A first; B is optional if the session time allows.

In [ ]:
import os
if not os.path.exists('/content/newsroom-agents'):
    !git clone https://github.com/Juanangara/newsroom-agents.git
%cd /content/newsroom-agents
!git pull -q
!pip install -q pydantic feedparser requests trafilatura googlenewsdecoder accelerate bitsandbytes transformers

In [ ]:
import sys
sys.path.insert(0, '/content/newsroom-agents/src')
from newsroom.benchmark import run_benchmark

MODELS = [
    'Qwen/Qwen2.5-3B-Instruct',
    'Qwen/Qwen2.5-7B-Instruct',
    'microsoft/Phi-3.5-mini-instruct',
]
VERIFIER = 'Qwen/Qwen2.5-7B-Instruct'
SNAPSHOTS = ['data/ia_colombia.json', 'data/reforma_pensional.json']

## A. Fixed verifier (run this first)

In [ ]:
res_fixed = run_benchmark(MODELS, SNAPSHOTS, verifier_id=VERIFIER, tag='fixed_verifier')

## B. Self-verification (optional)

In [ ]:
res_self = run_benchmark(MODELS, SNAPSHOTS, verifier_id=None, tag='self')

## Why were facts dropped?
Counts per model: `no_quote`, `quote_not_in_source`, `details_missing`, `headline_echo`.

In [ ]:
import json, glob
for f in sorted(glob.glob('results/benchmark_*.json')):
    print(f)
    for r in json.load(open(f)):
        print('  ', r['model'].split('/')[-1], '| verifier:', r['verifier'].split('/')[-1], '|', r['drop_reasons'])

Download the results before the session ends.

In [ ]:
from google.colab import files
!zip -qr results.zip results
files.download('results.zip')